In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/processed/customer_features.csv")
df.head()

,customer_id,recency_days,frequency,monetary,avg_order_value,total_interactions,last_interaction_days,interaction_types_count,engagement_segment,has_purchased,engaged_not_purchased,purchase_frequency_rate,interaction_purchase_ratio,customer_segment,location,acquisition_channel,age,gender
0,C00001,21,31,482863.71,15576.248710,335,7,4,High Engagement,1,0,0,10,medium,Bhubaneswar,ads,25,male
1,C00002,9,45,820236.81,18227.484667,389,4,4,High Engagement,1,0,0,8,high,Hyderabad,referral,47,female
2,C00003,550,0,0.00,0.000000,8,517,4,Medium Engagement,0,1,0,0,low,Ahmedabad,organic,46,female
3,C00004,10,31,537397.24,17335.394839,319,9,4,High Engagement,1,0,0,10,medium,Bangalore,organic,38,male
4,C00005,4,95,1556310.10,16382.211579,752,0,4,High Engagement,1,0,0,7,high,Hyderabad,referral,37,female


In [3]:
df.shape

(8000, 18)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customer_id                 8000 non-null   object 
 1   recency_days                8000 non-null   int64  
 2   frequency                   8000 non-null   int64  
 3   monetary                    8000 non-null   float64
 4   avg_order_value             8000 non-null   float64
 5   total_interactions          8000 non-null   int64  
 6   last_interaction_days       8000 non-null   int64  
 7   interaction_types_count     8000 non-null   int64  
 8   engagement_segment          8000 non-null   object 
 9   has_purchased               8000 non-null   int64  
 10  engaged_not_purchased       8000 non-null   int64  
 11  purchase_frequency_rate     8000 non-null   int64  
 12  interaction_purchase_ratio  8000 non-null   int64  
 13  customer_segment            8000 

## Create Target Features

In [7]:
# Define threshold
threshold = df["recency_days"].quantile(0.8)

# Create churn column
df["is_churned"] = (df["recency_days"] > threshold).astype(int)

# Check distribution
df["is_churned"].value_counts(normalize=True)

is_churned
0    0.8
1    0.2
Name: proportion, dtype: float64

In [13]:
# Create purchase target
df["will_purchase"] = df["has_purchased"]

In [14]:
df.columns

Index(['customer_id', 'recency_days', 'frequency', 'monetary',
       'avg_order_value', 'total_interactions', 'last_interaction_days',
       'interaction_types_count', 'engagement_segment', 'has_purchased',
       'engaged_not_purchased', 'purchase_frequency_rate',
       'interaction_purchase_ratio', 'customer_segment', 'location',
       'acquisition_channel', 'age', 'gender', 'is_churned', 'will_purchase'],
      dtype='object')

## Drop Leakage & Unnecessary Columns

In [15]:
columns_to_drop = [
    "customer_id",
    "recency_days",
    "last_interaction_days",
    "has_purchased"
]

df = df.drop(columns=columns_to_drop)

In [16]:
df.columns

Index(['frequency', 'monetary', 'avg_order_value', 'total_interactions',
       'interaction_types_count', 'engagement_segment',
       'engaged_not_purchased', 'purchase_frequency_rate',
       'interaction_purchase_ratio', 'customer_segment', 'location',
       'acquisition_channel', 'age', 'gender', 'is_churned', 'will_purchase'],
      dtype='object')

In [ ]:
# df.to_csv("../data/processed/modeling_dataset.csv", index=False)

## Prepare Data for Modeling

In [19]:
# Separate features and targets
X = df.drop(columns=["is_churned", "will_purchase"])
y_churn = df["is_churned"]
y_purchase = df["will_purchase"]

## Train-Test Split

In [20]:
from sklearn.model_selection import train_test_split

# Split using X and churn target
X_train, X_test, y_train_churn, y_test_churn = train_test_split(
    X, y_churn, test_size=0.2, random_state=42
)

# Align purchase target with the SAME split
y_train_purchase = y_purchase.loc[X_train.index]
y_test_purchase = y_purchase.loc[X_test.index]

In [ ]:
# Quick Validation
print(X_train.shape, X_test.shape)
print(y_train_churn.shape, y_test_churn.shape)
print(y_train_purchase.shape, y_test_purchase.shape)

(6400, 14) (1600, 14)
(6400,) (1600,)
(6400,) (1600,)


## Encode Categorical Variables

In [22]:
categorical_cols = [
    "customer_segment",
    "engagement_segment",
    "location",
    "acquisition_channel",
    "gender"
]

X_train_encoded = pd.get_dummies(
    X_train,
    columns=categorical_cols,
    drop_first=True
)

In [23]:
X_train_encoded.shape

(6400, 27)

In [24]:
X_test_encoded = pd.get_dummies(
    X_test,
    columns=categorical_cols,
    drop_first=True
)

In [ ]:
X_test_encoded.shape

(1600, 27)

## Align Train & Test

In [26]:
X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded,
    join='left',
    axis=1,
    fill_value=0
)

In [27]:
print(X_train_encoded.shape)
print(X_test_encoded.shape)

print((X_train_encoded.columns == X_test_encoded.columns).all())

(6400, 27)
(1600, 27)
True


## Scaling

In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

## Train Churn Model (Logistic Regression)

In [ ]:
from sklearn.linear_model import LogisticRegression

churn_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

churn_model.fit(X_train_scaled, y_train_churn)

LogisticRegression(class_weight='balanced', max_iter=1000)

## Evaluate Churn Model

In [34]:
from sklearn.metrics import classification_report, roc_auc_score

# Predictions
y_pred_churn = churn_model.predict(X_test_scaled)
y_prob_churn = churn_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("Classification Report:")
print(classification_report(y_test_churn, y_pred_churn))
print("ROC-AUC Score:", roc_auc_score(y_test_churn, y_prob_churn))

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.24      0.38      1273
           1       0.23      0.87      0.36       327

    accuracy                           0.37      1600
   macro avg       0.55      0.56      0.37      1600
weighted avg       0.75      0.37      0.37      1600

ROC-AUC Score: 0.5619632402929822


**I prioritized recall over accuracy because missing a churner is more costly than falsely targeting a non-churner.**

But there's a deeper issue: ROC-AUC ≈ 0.56    
That means, Model has weak separation power even after fixing imbalance.

Let's not tweak endlessly. We move to a better model for non-linear patterns now.

## Train Random Forest

In [35]:
from sklearn.ensemble import RandomForestClassifier

rf_churn_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf_churn_model.fit(X_train_encoded, y_train_churn)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [36]:
# Predictions
y_pred_rf = rf_churn_model.predict(X_test_encoded)
y_prob_rf = rf_churn_model.predict_proba(X_test_encoded)[:, 1]

# Evaluation
print("Random Forest - Churn Model:")
print(classification_report(y_test_churn, y_pred_rf))
print("ROC-AUC Score:", roc_auc_score(y_test_churn, y_prob_rf))

Random Forest - Churn Model:
              precision    recall  f1-score   support

           0       0.80      0.82      0.81      1273
           1       0.24      0.22      0.23       327

    accuracy                           0.70      1600
   macro avg       0.52      0.52      0.52      1600
weighted avg       0.69      0.70      0.69      1600

ROC-AUC Score: 0.575030448914289


- If goal = "Don't miss churners" (Retention use-case)
    - Logistic Regression wins
        - Because missing a churner = lost customer = lost revenue
- If goal = "Avoid wasting money"
    - Random Forest is slightly better (but still weak)

**The models show limited predictive power, suggesting that the current feature set does not strongly separate churners from non-churners.**

## Feature Importance

In [39]:
feature_importance = pd.DataFrame({
    "feature": X_train_encoded.columns,
    "coefficient": churn_model.coef_[0]
})

feature_importance = feature_importance.sort_values(by="coefficient", ascending=False)
feature_importance

,feature,coefficient
13,engagement_segment_No Interaction,0.280871
4,interaction_types_count,0.277655
2,avg_order_value,0.148311
10,customer_segment_medium,0.087860
12,engagement_segment_Medium Engagement,0.073336
9,customer_segment_low,0.037241
18,location_Hyderabad,0.015168
26,gender_other,0.013328
15,location_Bhubaneswar,0.001896
6,purchase_frequency_rate,0.000000


Churn is primarily driven by disengagement. Customers with no interaction or poor interaction patterns are more likely to leave. On the other hand, strong purchase behavior, such as high frequency and monetary value, significantly reduces churn risk. This indicates that engagement quality and transactional activity are the key drivers, while demographic factors have minimal influence.

## Create Purchase Feature Set

In [43]:
purchase_drop_cols = [
    "frequency",
    "monetary",
    "avg_order_value",
    "purchase_frequency_rate",
    "interaction_purchase_ratio",
    "engaged_not_purchased"
]

X_train_purchase = X_train_encoded.drop(columns=purchase_drop_cols)
X_test_purchase = X_test_encoded.drop(columns=purchase_drop_cols)

In [44]:
X_train_purchase.shape

(6400, 21)

## Scale Purchase Features

In [45]:
from sklearn.preprocessing import StandardScaler

scaler_purchase = StandardScaler()

X_train_purchase_scaled = scaler_purchase.fit_transform(X_train_purchase)
X_test_purchase_scaled = scaler_purchase.transform(X_test_purchase)

## Train Purchase Model

In [46]:
from sklearn.linear_model import LogisticRegression

purchase_model = LogisticRegression(max_iter=1000)
purchase_model.fit(X_train_purchase_scaled, y_train_purchase)

LogisticRegression(max_iter=1000)

## Evaluate Purchase Model

In [47]:
from sklearn.metrics import classification_report, roc_auc_score

# Predictions
y_pred_purchase = purchase_model.predict(X_test_purchase_scaled)
y_prob_purchase = purchase_model.predict_proba(X_test_purchase_scaled)[:, 1]

# Evaluation
print("Purchase Model Performance:")
print(classification_report(y_test_purchase, y_pred_purchase))
print("ROC-AUC Score:", roc_auc_score(y_test_purchase, y_prob_purchase))

Purchase Model Performance:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1388
           1       0.93      0.94      0.94       212

    accuracy                           0.98      1600
   macro avg       0.96      0.97      0.96      1600
weighted avg       0.98      0.98      0.98      1600

ROC-AUC Score: 0.9985862650209343


**The model shows strong predictive power, largely because purchase behavior is closely tied to engagement patterns in the dataset.**